# 02 · Turning transactions into tokens

**NVIDIA TFM Blueprint, notebook 2 of 5.** A language model needs tokens. This
notebook shows how one transaction becomes 11 field tokens and a separator, how
a customer's recent history becomes one row, and how the pretraining corpus is
cut.

Every notebook runs on CPU in about a minute on a small generated book, so
you can read the whole series without downloading anything. To see real rows,
build the TabFormer splits once (notebook 01 shows how) and set
`TFM_TABFORMER_ROOT`; the notebooks then load 24 TabFormer users with
`config.small.yaml`. The full-size runs are recipe commands, listed in notebooks 03
and 05, not notebook cells.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os

from tfm.config import load_recipe_config
from tfm.data.adapters.paths import ADAPTER_ENV
from tfm.extras.capability import extra_available
from tfm.runtime import prepare_data
from tfm.runtime.common import recipe_dir
from tfm.training.reproducibility import seed_everything

USING_TABFORMER = bool(os.environ.get(ADAPTER_ENV["tabformer"]))
CONFIG = recipe_dir("blueprint_v1") / ("config.small.yaml" if USING_TABFORMER else "config.smoke.yaml")
config = load_recipe_config(CONFIG)
if config.model.backbone == "huggingface" and not extra_available("transformers"):
    print("transformers extra not installed; using the eager decoder (uv sync --extra native --extra huggingface)")
    composition = config.model.composition.model_copy(update={"position": "nope"})
    model_cfg = config.model.model_copy(update={"backbone": "eager", "composition": composition})
    config = config.model_copy(update={"model": model_cfg})
seed_everything(config.seed)
prepared = prepare_data(config)
tokenizer = prepared.tokenizer
train_batch, val_batch, test_batch = prepared.batches
print("config", CONFIG.name, "| adapter", config.data.adapter, "| backbone", config.model.backbone)

## 1. Eleven fields, eleven tokens

Each transaction becomes one token per field, always in the same order:
amount bucket, merchant, merchant category, MCC, hour, weekday, month, card
index, chip use, ZIP3, and state. The vocabulary is fixed in advance, so the
tokenizer needs no fitting to the data. `features.fields` picks the fields.

In [ ]:
from tfm.features.tokenizers.financial import FIELD_ORDER, SPECIAL_TOKENS

history = prepared.train_sequences[0]
print("v1's fields ", FIELD_ORDER)
print("this recipe", tokenizer.fields)
print("special tokens", SPECIAL_TOKENS)
print("vocabulary", tokenizer.vocab_size)
print("one transaction:", tokenizer.event_tokens(history.events[0]))

## 2. Why the customer token is gone

v1 had a twelfth field, `CUST`, the customer id. Inside one customer's
history it is the same token at every transaction: free to predict, so it
pads the loss with an easy target, and it hands the model the customer's
identity, which a fraud model should not key on. Tokenize the same history
with v1's twelve fields to see it.

In [ ]:
from tfm.features.tokenizers.financial import TransactionFieldTokenizer

v1_fields = TransactionFieldTokenizer()
cust = {v1_fields.event_tokens(e)[FIELD_ORDER.index("CUST")] for e in history.events}
print(len(history.events), "transactions, distinct CUST tokens:", cust)

## 3. A history is a row of tokens

A history is `<bos>`, then each transaction's 11 tokens, a `<sep>` between
transactions, and `<eos>`: 12 tokens per transaction plus one. Shorter rows are
padded, and the attention mask marks the padding so the model ignores it. A
row that is too long keeps `<bos>` and the most recent whole transactions.

In [ ]:
encoded = tokenizer.encode(history)
n = len(history.events)
print("transactions", n, "tokens", len(encoded.token_ids), "= 12 *", n, "+ 1:", 12 * n + 1 == len(encoded.token_ids))
print("first 14 tokens:", [tokenizer.id_to_token[i] for i in encoded.token_ids[:14]])
ids, mask = tokenizer.encode_sequence(history, max_length=1 + 12 * 3)
kept = [i for i, m in zip(ids, mask) if m]
same = kept[1:] == list(encoded.token_ids[-(len(kept) - 1):])
print("cut to", len(ids), "tokens:", kept.count(tokenizer.sep_id) + 1, "transactions, the most recent:", same)

## 4. The row behind each scored transaction

Notebook 01's rows are transactions. Each one is tokenized as the customer's
last `data.sequence_length` transactions up to and including it, so the
transaction being scored is always the last one in its row. Beside it, the
kit keeps v1's version of the same transaction: the transaction alone.

In [ ]:
from tfm.data.sequence_builders.event_rows import build_event_history_rows
from tfm.runtime.common import parse_duration

scored = build_event_history_rows(
    prepared.events,
    entities=sorted({e.entity_id for e in prepared.events})[:4],
    window=prepared.cutoffs.windows["train"],
    label_cutoff=prepared.cutoffs.train,
    max_events=config.data.sequence_length,
    observation_window=parse_duration(config.data.observation_window),
    max_rows=3,
    seed=config.seed,
)
for row in scored:
    last = row.history.events[-1]
    print(
        f"{row.history.sequence_id}: {len(row.history.events)} transactions, ends at {last.event_timestamp:%Y-%m-%d %H:%M},",
        f"label {row.label:.0f}, alone: {len(row.event_only.events)} transaction",
    )

## 5. The pretraining corpus

Pretraining rows are wider than scoring rows. The full-size config cuts each
customer's training-period history into windows of 341 transactions, which is
12 × 341 + 1 = 4,093 tokens, one row of up to 4,096. Windows never cross
customers.

In [ ]:
full = load_recipe_config(CONFIG.with_name("config.yaml")).data
per_event = len(tokenizer.fields) + 1
print("window", full.pretrain_windows.events, "transactions =", per_event * full.pretrain_windows.events + 1, "tokens")
print("scoring rows", full.sequence_length, "transactions =", full.max_tokens, "tokens | users", full.n_entities)
print("this run:", len(prepared.pretrain_sequences), "pretraining rows")

> **What changed from v1**
>
> - **Eleven fields, not twelve.** `CUST` is dropped (section 2).
> - **CPU merchant hash.** v1 bucketed merchants with cuDF's hash, which has
>   since changed, so no current library reproduces its buckets. The kit hashes
>   on CPU; `rapids_field_tokenizer` hashes on GPU.
> - **One id, one token.** v1's vocabulary gave `MONTH_12` and `CARD_0` the same
>   id, so the model couldn't tell December from the first card. The kit
>   assigns ids densely (checked below).
> - **Windows per customer.** v1 cut windows per card; here windows are per
>   customer, the same unit as the rows being scored.
> - **No corpus files.** v1 wrote token text files for NeMo. The kit tokenizes in
>   memory.

In [ ]:
print("MONTH_12", tokenizer.token_to_id["MONTH_12"], "CARD_0", tokenizer.token_to_id["CARD_0"])
assert tokenizer.token_to_id["MONTH_12"] != tokenizer.token_to_id["CARD_0"]

## Takeaways

- One transaction is 11 field tokens; a history adds `<bos>`, `<sep>`s, and
  `<eos>`: 12 tokens per transaction plus one.
- Each scored transaction is the last one in its own history row.
- Pretraining rows are longer windows of the same customers' histories.

Code: `tfm.features.tokenizers.financial`, `tfm.data.sequence_builders`. The
key-value-time tokenizer of Blueprint v2 is a different design; see
[`notebooks/pragmatiq`](../pragmatiq).

Next: [03 · Pretraining](03_foundation_model_training.ipynb).